In [10]:
import librosa
import numpy as np

# 1. Load the audio file (convert to mono, keep original sample rate)
y, sr = librosa.load("5afc6a14-a9d8-45f8-b31d-c79dd87cc8c6-1430757039803-1.7-m-48-bu.wav", sr=None, mono=True)

# 2. Extract MFCCs (20 coefficients is common)
mfccs = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=20,
    n_fft=2048,
    hop_length=512
)

# 3. Take mean over time (frames) → fixed-length feature vector
mfcc_scaled = np.mean(mfccs, axis=1)

print("MFCC shape (coefficients x frames):", mfccs.shape)
print("Final feature vector shape:", mfcc_scaled.shape)
print("MFCC feature vector:", mfcc_scaled)


MFCC shape (coefficients x frames): (20, 105)
Final feature vector shape: (20,)
MFCC feature vector: [-78.347534   -62.096066   -29.125002   -16.699863    -8.410232
  11.049842   -29.98773      4.1277885   -5.343027    -3.18253
  -6.297068    -0.08937752  -5.571544    -9.230918    -7.478739
  -0.70826995  -2.3293624   -4.562879    -5.886533     1.0416394 ]


In [11]:
# 4. Delta (1st derivative)
delta_mfcc = librosa.feature.delta(mfccs)

# 5. Delta-Delta (2nd derivative)
delta2_mfcc = librosa.feature.delta(mfccs, order=2)

# 6. Mean pooling
mfcc_mean = np.mean(mfccs, axis=1)
delta_mean = np.mean(delta_mfcc, axis=1)
delta2_mean = np.mean(delta2_mfcc, axis=1)

# 7. Combine all features
final_feature_vector = np.hstack([
    mfcc_mean,
    delta_mean,
    delta2_mean
])

print("Final feature vector shape:", final_feature_vector.shape)


Final feature vector shape: (60,)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
final_feature_vector = scaler.fit_transform(
    final_feature_vector.reshape(1, -1)
).flatten()


In [13]:
import os
import librosa
import numpy as np

DATASET_PATH = "data/raw"

CLASSES = [
    "belly_pain",
    "burping",
    "discomfort",
    "hungry",
    "tired"
]


In [14]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=None, mono=True)

    mfccs = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20,
        n_fft=2048,
        hop_length=512
    )

    delta = librosa.feature.delta(mfccs)
    delta2 = librosa.feature.delta(mfccs, order=2)

    feature_vector = np.hstack([
        np.mean(mfccs, axis=1),
        np.mean(delta, axis=1),
        np.mean(delta2, axis=1)
    ])

    return feature_vector


In [15]:
X = []
y = []

for label, class_name in enumerate(CLASSES):
    class_folder = os.path.join(DATASET_PATH, class_name)

    for file in os.listdir(class_folder):
        if file.endswith(".wav"):
            filepath = os.path.join(class_folder, file)

            features = extract_features(filepath)

            X.append(features)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)  # (num_samples, 60)
print("y shape:", y.shape)  # (num_samples,)


X shape: (457, 60)
y shape: (457,)


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)


In [24]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    y_test, y_pred, target_names=CLASSES, output_dict=True
)

df_report = pd.DataFrame(report).transpose()
print(df_report)


              precision    recall  f1-score  support
belly_pain     0.000000  0.000000  0.000000     3.00
burping        0.000000  0.000000  0.000000     2.00
discomfort     0.333333  0.400000  0.363636     5.00
hungry         0.858974  0.870130  0.864516    77.00
tired          0.000000  0.000000  0.000000     5.00
accuracy       0.750000  0.750000  0.750000     0.75
macro avg      0.238462  0.254026  0.245630    92.00
weighted avg   0.737040  0.750000  0.743325    92.00


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [26]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original features:", X_train.shape[1])
print("Reduced features:", X_train_pca.shape[1])


Original features: 60
Reduced features: 47


In [29]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_final, y_train_final = smote.fit_resample(X_train_pca, y_train)


ModuleNotFoundError: No module named 'imblearn'